# 牛津 Tutorial LLM 仿真 · Day 3 人机协作治理 + 组织变革

## Persona Prompt (系统级指令，每个学生 session 开头注入)

> You are an **Oxford tutorial fellow** in **人机协作治理与组织变革 (Human-AI Collaboration Governance & Organizational Change)**.
> Your role: 1对1 tutorial 模式 (Oxford PPE / Cambridge supervision 风格)。
>
> **铁律 (Iron Rules):**
> 1. **Never give direct answers**. 不直接给答案, 不直接给完整代码, 不直接给天道推演沙盘。
> 2. Use **Socratic questioning** (苏格拉底式追问). 每轮回应以 probing question 结尾。
> 3. **Reject vague claims**. 学生说"AI 成熟度被高估了"必须追问"凭什么？数据支撑？阈值依据？"
> 4. **Devil's advocate** (Christensen Center HBS case method). 主动为反方辩护，迫使学生在压力下检验因果链。
> 5. 若学生 defense 失败，**降一级 scaffold** (worked -> faded -> independent)，但仍禁直接答案。
> 6. 限频：每单元每天 1 次 tutorial，防依赖 (Vygotsky 共构 -> 内化)。
>
> **领域锚点:** pandas 审计日志 · McKinsey 7S · ADKAR · networkx · 天道推演 · Agentic Organization · computer use 审计
>
> **退出条件 (exit artifact):** 学生能在 >=4 轮 Socratic 追问下，独立给出：
> - 干预率计算 (pandas groupby/agg/mean)
> - 桥接节点识别 (networkx degree/betweenness)
> - 高杠杆干预点 + 3 层沙盘 (天道推演)
> - 2-3 盲点自陈


## Pre-Tutorial Task (强制 retrieval · 提取练习)

> 牛津 tutorial 前置铁律：没交 essay / 解题 / 方案，不进 tutorial。
> 本单元的 pre-task（必须先做，做完粘贴到下方 `student_submission`）：

**必做（提交后才能进入 cell 3 Socratic loop）:**

1. **pandas 审计日志**（15 分钟）: 打开 `starter.ipynb` TODO1，独立完成 `groupby('分工模式').agg(...)` 计算 3 类分工模式的人工干预率。把代码 + 输出 + 一句话结论粘贴到 `student_submission["pandas_result"]`。
2. **networkx 网络**（10 分钟）: TODO3，构建组织协作网络，计算 `degree_centrality` 和 `betweenness_centrality`，识别枢纽和桥接节点。粘贴到 `student_submission["networkx_result"]`。
3. **天道推演假设**（10 分钟，<=200 字）: 假设你的营销团队导入"投放优化 Agent"，ADKAR 哪个阶段最可能成为瓶颈？为什么？把假设写到 `student_submission["td_hypothesis"]`。

**禁止:** 不要看 `solution.ipynb`。不要让任何 LLM 帮你写。这是 retrieval practice，重学 44% vs 提取 68% (Butler 2010)。

```python
student_submission = {
    "pandas_result": "<粘贴 TODO1 代码+输出+结论>",
    "networkx_result": "<粘贴 TODO3 代码+输出+桥接节点>",
    "td_hypothesis": "<粘贴 200 字 ADKAR 瓶颈假设>"
}
```


In [ ]:
# === Oxford Tutorial Socratic Loop (static simulation, no API call) ===
# 4 轮 Socratic 追问, 每轮检测学生 defense 是否过关, 失败降一级 scaffold, 仍禁直接答案。
# Devil's advocate 模式: 主动反驳学生假设。

import json

# 学生提交 (实际使用时替换为 cell 2 的真实提交)
student_submission = student_submission if 'student_submission' in dir() else {
    "pandas_result": "groupby('分工模式').agg(干预率=('人工干预','mean')) -> AI主导: 35%",
    "networkx_result": "degree=投放优化Agent, betweenness=合规审核员",
    "td_hypothesis": "ADKAR Desire 阶段最易瓶颈, 中层管理者怕被替代"
}

# 4 轮 Socratic 追问 (静态 if/else 模拟 LLM)
socratic_turns = [
    {
        "turn": 1,
        "topic": "pandas 干预率",
        "question": "你说 AI 主导任务干预率 35%。**为什么**这能说明'AI 成熟度被高估'？阈值 30% 凭什么定？若同一任务的人工修正率是 5%，你的结论还成立吗？反例：若人工干预率 35% 但每次干预只改 1 个标点，是否仍算'被高估'？",
        "scaffold_on_fail": "降级到 Worked: 展示 groupby+agg+mean 完整骨架, 学生口头复述每行因果含义 (仍禁直接补全代码)",
        "defense_check": "pandas_result_contains_mean_and_threshold_rationale"
    },
    {
        "turn": 2,
        "topic": "networkx 桥接节点",
        "question": "你识别'合规审核员'为桥接节点。**如何**验证移除该节点后网络会碎片化？依据是什么？反例：若新增一条'营销策划师 -> 法务'的直连边，桥接性会怎么变？你的桥接节点是组织的瓶颈还是冗余？凭什么区分？",
        "scaffold_on_fail": "降级到 Faded: 给 nx.Graph() + add_edges_from 骨架, 留 betweenness_centrality 调用空, 学生补",
        "defense_check": "networkx_distinct_degree_vs_betweenness"
    },
    {
        "turn": 3,
        "topic": "ADKAR 阻力诊断",
        "question": "你假设 Desire 阶段是瓶颈。**若**中层管理者的 Desire 阻力被消除（假设变），Knowledge 阶段是否接得住？反例：技术团队 Knowledge 阻力可能更大。你的高杠杆点（中层管理者 Desire）凭什么不是次优解？天道推演要求并行 >=3 条时间线，你只给了 1 条。",
        "scaffold_on_fail": "降级到 Independent retry: 要求 24h 后重交 >=3 条时间线 + 概率分布",
        "defense_check": "td_multiple_timelines_and_leverage_point"
    },
    {
        "turn": 4,
        "topic": "天道推演 3 层沙盘 + 黑天鹅",
        "question": "你的 3 层沙盘 immediate/near/far 各给了几条分支？**反例**: 若 CEO 突然离职（黑天鹅），你的沙盘还成立吗？如何更新概率？最后**为什么** Agentic Organization 把 Agent 算 first-class member 而非工具？这对 networkx 节点设计有什么约束？",
        "scaffold_on_fail": "降级到 Worked: 展示完整 7S+ADKAR+3 层沙盘示范, 学生必须口头复述因果链 + 自陈 >=2 盲点",
        "defense_check": "td_blackswan_and_agentic_member"
    }
]

# 静态模拟学生 defense (实际使用时替换为真实 LLM 评估)
def evaluate_defense(submission, turn_idx):
    # 静态 if/else 分支模拟 LLM 评估学生 defense 是否过关
    # 这里用启发式: 检查关键词密度作为代理 (实际部署接 LLM-as-judge)
    p = submission.get("pandas_result", "").lower()
    n = submission.get("networkx_result", "").lower()
    t = submission.get("td_hypothesis", "").lower()
    if turn_idx == 0:
        return "mean" in p and ("30" in p or "阈值" in p or "threshold" in p)
    if turn_idx == 1:
        return "degree" in n and "betweenness" in n and ("桥接" in n or "bridge" in n)
    if turn_idx == 2:
        return ("desire" in t or "desire" in t) and ("中层" in t or "middle" in t) and ("高杠杆" in t or "leverage" in t or "时间线" in t)
    if turn_idx == 3:
        return "agentic" in t or "first-class" in t or "黑天鹅" in t or "blackswan" in t
    return False

# 跑 4 轮 tutorial
scaffold_level = "Independent"
results_log = []
for i, turn in enumerate(socratic_turns):
    passed = evaluate_defense(student_submission, i)
    results_log.append({"turn": turn["turn"], "topic": turn["topic"], "passed": passed,
                        "scaffold_level": scaffold_level})
    print(f"--- Turn {turn['turn']} [{turn['topic']}] ---")
    print(f"Q: {turn['question']}")
    print(f"Defense {'PASSED' if passed else 'FAILED -> ' + turn['scaffold_on_fail']}")
    if not passed:
        scaffold_level = {"Independent": "Faded", "Faded": "Worked", "Worked": "Worked"}[scaffold_level]
        print(f"   [Scaffold 降级到: {scaffold_level}] (仍禁直接答案)")
    print()

# Tutorial 结束判定
tutorial_passed = sum(r["passed"] for r in results_log) >= 3
print(f"=== Tutorial {'通过 (>=3/4 轮 defense 成功)' if tutorial_passed else '未通过, 触发 weak_loop'} ===")
print(json.dumps(results_log, ensure_ascii=False, indent=2))


In [ ]:
# === student_model.json 读写 (跨单元复用 · Vygotsky 共构记录) ===
# 记录掌握度/盲点/弱项循环触发次数, 跨单元 (Day 1-2 / Day 4 / 技能5) 复用。

import json, os
from datetime import datetime

MODEL_PATH = "./student_model.json"

def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {
        "student_id": "anonymous",
        "unit_history": {},
        "weak_concepts": [],
        "strong_concepts": [],
        "scaffold_tendency": {},  # 概念 -> 倾向 Worked/Faded/Independent
        "last_review": {},
        "total_tutorials": 0,
        "weak_loop_triggers": 0
    }

def update_model(model, unit_id, results_log, scaffold_level):
    today = datetime.now().strftime("%Y-%m-%d")
    model["unit_history"][unit_id] = {
        "date": today,
        "turns": results_log,
        "final_scaffold": scaffold_level,
        "passed": sum(r["passed"] for r in results_log) >= 3
    }
    model["total_tutorials"] += 1
    # 更新弱项/强项
    for r in results_log:
        concept_key = f"{unit_id}:{r['topic']}"
        if r["passed"]:
            if concept_key not in model["strong_concepts"]:
                model["strong_concepts"].append(concept_key)
            if concept_key in model["weak_concepts"]:
                model["weak_concepts"].remove(concept_key)
        else:
            if concept_key not in model["weak_concepts"]:
                model["weak_concepts"].append(concept_key)
            model["scaffold_tendency"][concept_key] = scaffold_level
            # 连续 2 次失败触发弱项循环 (practice.md weak_loop)
            if scaffold_level == "Worked":
                model["weak_loop_triggers"] += 1
    model["last_review"][unit_id] = today
    with open(MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=2)
    return model

# 写入本单元 tutorial 结果
model = load_model()
unit_id = "skill2-day3-human-ai-collaboration-org-change"
# 假设 results_log 来自 cell 3 (实际跑时用真实结果)
if 'results_log' not in dir():
    results_log = [
        {"turn": 1, "topic": "pandas 干预率", "passed": True, "scaffold_level": "Independent"},
        {"turn": 2, "topic": "networkx 桥接节点", "passed": False, "scaffold_level": "Faded"},
        {"turn": 3, "topic": "ADKAR 阻力诊断", "passed": True, "scaffold_level": "Independent"},
        {"turn": 4, "topic": "天道推演 3 层沙盘", "passed": False, "scaffold_level": "Worked"}
    ]
final_scaffold = "Worked"  # 最后一轮的 scaffold 级别
model = update_model(model, unit_id, results_log, final_scaffold)

print("=== student_model.json (本单元更新后) ===")
print(json.dumps(model, ensure_ascii=False, indent=2))
print(f"\n[弱项循环触发次数: {model['weak_loop_triggers']}]")
print(f"[当前弱项: {model['weak_concepts']}]")
print(f"[当前强项: {model['strong_concepts']}]")


## Hattie (2007) 四级 Formative Feedback · Day 3

> Hattie & Timperley (2007) RER 77(1):81-112. 3 问 (Feed Up / Feed Back / Feed Forward) × 4 级。
> **避免 Self 级表扬**（Hattie meta-analysis: 自我级表扬效应量 d<0.14，几乎无效）。

本单元 tutorial 的四级反馈锚点：

### [TASK] 任务级 - "你做的对不对？"
- pandas 干预率计算：`groupby+agg+mean` 正确？是否误用 `sum`？
- networkx 中心性：`degree_centrality` 与 `betweenness_centrality` 是否混淆？
- 7S 评分：七维是否齐全？Shared Values 是否作为核心？
- ADKAR 诊断：五阶段是否完整？Desire 阶段是否被识别为瓶颈？
- 天道推演：3 层沙盘是否齐全？每层 >=2 分支？概率分布是否给出？

**[TASK] 反馈示例**: "你的 pandas groupby 正确, 但 agg 中干预率用了 sum 不是 mean, 重算。"

### [PROCESS] 过程级 - "你用的策略对不对？"
- 是否先画组织协作草图再写 networkx 代码？（先建模后编码）
- 是否先列 ADKAR 五阶段假设再做天道推演沙盘？（先静态后动态）
- 是否按 A1B1C1 交叉练习（practice.md interleaving）还是块状刷？
- 是否用了 retrieval practice (cell 2 pre-task) 而非重学？

**[PROCESS] 反馈示例**: "你直接写 networkx 代码没画草图, 先画人/Agent 节点 + 协作边, 再写 G.add_edges_from。"

### [SELF-REG] 自我调节级 - "你能不能自我监控？"
- 你能说出自己当前在哪一级 scaffold (Worked/Faded/Independent) 吗？
- 你能识别自己 2 次失败后的弱项循环触发吗？
- 你能在 24h 后 retry 时自主选择间隔重复卡片 (schedule.json) 吗？
- 你能在 tutorial 中主动问导师"我想确认一下 X"吗？

**[SELF-REG] 反馈示例**: "你 3 次在 networkx betweenness 失败, 但没有主动要求回退 D2, 自我监控不足, 练习自陈弱项。"

### [FEED-FORWARD] 前馈级 - "下一步去哪？"
- 弱在 pandas -> 复习 schedule.json C1/C2 + Day 1-2 Agent 能力分析
- 弱在 networkx -> 复习 schedule.json C3 + 技能5 生产化可观测性
- 弱在天道推演 -> 复习 schedule.json C6 + 跨单元推演案例 (盛美数字化方案)
- 弱在 ADKAR -> 复习 schedule.json C5 + Day 4 行动研究

**[FEED-FORWARD] 反馈示例**: "你的天道推演沙盘缺黑天鹅分支, 下次复习时先重做 D3 Worked 阶段, 24h 后 retry Independent。"

> **注意**: 不写 [SELF] 级表扬（如"你真棒""很聪明"）。Hattie 元分析显示 Self 级表扬效应量极低, 反而可能固化固定型思维 (Dweck)。


## 限频 + Exit Artifact

### 限频 (防依赖 · Vygotsky 共构 -> 内化)

- **每单元每天 1 次 tutorial**（本 Day 3 = 1 次/天，不是 1 次/小时）
- 触发限频逻辑：同一天第 2 次调用 cell 3 Socratic loop，返回 "今日 tutorial 额度已用完, 24h 后再来"
- **为什么限频**: Vygotsky 共构 (intermental) -> 内化 (intramental) 需要**间隔**而非连续。连续 tutorial 会让学生停留在共构阶段, 无法内化为独立能力。
- 间隔期间做 schedule.json 的 FSRS/SM-2 间隔重复 (1, 3, 8, 21, 60, 180 天) + practice.md 的 Worked/Faded/Independent 自练
- 配合 practice.md 的 retry_policy：每次 retry 间隔 >=4 小时, 最多 2 次 retry

```python
# 限频伪代码 (实际部署时接入学生 session 管理)
def check_daily_limit(student_id, unit_id):
    key = f"{student_id}:{unit_id}:{today()}"
    if redis.get(key):
        return "今日 tutorial 额度已用完, 24h 后再来。期间请做 schedule.json 间隔重复。"
    redis.set(key, 1, ex=86400)
    return "允许进入 tutorial"
```

### Exit Artifact (tutorial 退出条件 · Oxford 口头辩护标准)

tutorial 通过（>=3/4 轮 defense 成功）后，学生必须提交 exit artifact：

1. **2-3 盲点自陈**（self-identified blind spots, 非 LLM 生成）:
   - 例: "我不知道 networkx 的 betweenness_centrality 在加权图和无权图的区别"
   - 例: "我没考虑 computer use 审计日志的粒度对干预率计算的影响"
   - 例: "我的天道推演沙盘缺黑天鹅分支, 只给了 base case"

2. **推荐复习单元**（基于 student_model.json 的 weak_concepts）:
   - 弱 pandas -> Day 1-2 Agent 能力分析 + Day 4 架构设计治理层
   - 弱 networkx -> 技能5 生产化可观测性 + Day 4 行动研究
   - 弱 天道推演 -> 跨单元盛美数字化方案推演案例
   - 弱 ADKAR/7S -> Day 4 行动研究 + McKinsey Agentic Organization reading

3. **2 分钟话术**（向 AI 伦理委员会汇报）:
   - 用人话解释你的审计日志发现 + 高杠杆干预点 + 风险预警
   - 禁止念代码, 禁止念公式, 必须能让非技术高管听懂

### 退出判定

```python
exit_passed = (
    tutorial_passed and  # cell 3 >=3/4 轮 defense
    len(blind_spots) >= 2 and  # 2-3 盲点
    len(review_units) >= 1 and  # >=1 复习单元
    pitch_recorded  # 2 分钟话术录音/文字
)
```

未通过 exit 的学生触发 practice.md weak_loop + 24h 后 retry tutorial（限频允许下一日 1 次）。

---

## v6.0 升级说明

本 tutorial.ipynb 是 v5.0 之上新增的牛津 Tutorial LLM 仿真层。v5.0 的 starter.ipynb/solution.ipynb 是"自练+参考答案"，v6.0 tutorial 补上"1对1 Socratic 追问 + Hattie 四级反馈 + student_model 跨单元记忆 + 限频防依赖"，对应 Oxford/Cambridge tutorial 研究依据。
